# Diffusion Model 演示

这个notebook演示如何使用DDPM（Denoising Diffusion Probabilistic Models）在MNIST数据集上训练和生成图像。


## 1. 导入必要的库


In [1]:
import torch
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# 导入我们的diffusion模型
from diffsuion_model import Config, DiffusionModel, get_dataloader, prepare_noise_schedule

print(f"PyTorch版本: {torch.__version__}")
print(f"可用设备: {torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')}")


ModuleNotFoundError: No module named 'diffsuion_model'

## 2. 可视化前向扩散过程

让我们看看图像是如何逐步变成噪声的。


In [ ]:
# 加载一张MNIST图像
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x * 2) - 1)
])

dataset = datasets.MNIST(root='../data', train=True, download=True, transform=transform)
image, label = dataset[0]

# 准备噪声调度
noise_schedule = prepare_noise_schedule(1000, 0.0001, 0.02)

# 可视化不同时间步
timesteps_to_show = [0, 100, 300, 500, 700, 999]
fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(15, 3))

noise = torch.randn_like(image)
for idx, t in enumerate(timesteps_to_show):
    sqrt_alpha = noise_schedule['sqrt_alphas_cumprod'][t]
    sqrt_one_minus = noise_schedule['sqrt_one_minus_alphas_cumprod'][t]
    noisy_image = sqrt_alpha * image + sqrt_one_minus * noise
    
    axes[idx].imshow(noisy_image.squeeze(), cmap='gray')
    axes[idx].set_title(f't={t}')
    axes[idx].axis('off')

plt.suptitle('前向扩散：从图像到噪声')
plt.tight_layout()
plt.show()


## 3. 训练Diffusion模型

**注意**: 为了快速演示，我们使用较少的训练轮数。


In [ ]:
# 配置
config = Config()
config.epochs = 5  # 快速演示
config.timesteps = 300
config.batch_size = 256

print(f"配置: {config.epochs} epochs, {config.timesteps} timesteps, device: {config.device}")

# 加载数据并训练
dataloader = get_dataloader(config)
diffusion = DiffusionModel(config)
diffusion.train(dataloader)


## 4. 从噪声生成图像

训练完成后，从纯噪声开始逐步去噪生成新的手写数字。


In [ ]:
# 生成样本
print("从噪声生成图像...")
samples, intermediate = diffusion.sample(batch_size=16)

# 显示生成的图像
samples = (samples + 1) / 2
grid = make_grid(samples.cpu(), nrow=4, padding=2)

plt.figure(figsize=(10, 10))
plt.imshow(grid.permute(1, 2, 0), cmap='gray')
plt.title('生成的手写数字', fontsize=16)
plt.axis('off')
plt.show()


## 5. 观察反向去噪过程


In [ ]:
# 显示中间去噪步骤
if intermediate:
    fig, axes = plt.subplots(1, len(intermediate), figsize=(20, 4))
    for idx, step_imgs in enumerate(intermediate):
        img = (step_imgs[0] + 1) / 2
        axes[idx].imshow(img.squeeze(), cmap='gray')
        step = config.timesteps - idx * 200
        axes[idx].set_title(f't={step}')
        axes[idx].axis('off')
    
    plt.suptitle('反向去噪：从噪声到清晰图像')
    plt.tight_layout()
    plt.show()
